# V2 Control Plane + Drive ZIP (Colab)
Notebook per proves mes serioses amb dataset gran des de Drive i worker server-driven.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!test -d b-ia && (cd b-ia && git pull) || git clone https://github.com/rserrar/bia.git b-ia
root_candidates = [Path('/content/b-ia'), Path('/content/b-ia/V2')]
REPO_ROOT = None
for candidate in root_candidates:
    if (candidate / 'colab-worker').exists() and (candidate / 'shared').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise FileNotFoundError('No s'ha pogut detectar l'arrel del repo (calen colab-worker/ i shared/).')
%cd {REPO_ROOT}
print('Repo root detectat:', REPO_ROOT)

In [ ]:
!pip -q install requests numpy pandas tensorflow google-genai

In [ ]:
import os

V2_API_BASE_URL = 'https://control.einavirtual.com'
V2_API_TOKEN = '<token-real>'
V2_API_PATH_PREFIX = '/public/index.php'

V2_LLM_API_KEY = '<openai-api-key>'
V2_LLM_MODEL = 'gpt-5.4'

GEMINI_API_KEY = ''  # opcional fallback
V2_LLM_FALLBACK_MODEL = 'gemini-3-flash-preview'

DATA_ZIP_PATH = '/content/drive/MyDrive/b-ia/dades/borsa_min.zip'
DATASET_RUNTIME_NAME = 'borsa_drive_runtime'
CLEAN_EXTRACT = False

if '<token-real>' in V2_API_TOKEN or '<openai-api-key>' in V2_LLM_API_KEY:
    raise RuntimeError('Configura V2_API_TOKEN i V2_LLM_API_KEY abans d\'executar.')

os.environ['PYTHONPATH'] = str(REPO_ROOT)
os.environ['V2_API_BASE_URL'] = V2_API_BASE_URL
os.environ['V2_API_TOKEN'] = V2_API_TOKEN
os.environ['V2_API_PATH_PREFIX'] = V2_API_PATH_PREFIX
os.environ['V2_CODE_VERSION'] = 'colab-drive-zip-control-plane'
os.environ['V2_CHECKPOINT_PATH'] = '/content/drive/MyDrive/bia_v2/run_state.json'
os.environ['V2_HEARTBEAT_INTERVAL_SECONDS'] = '30'
os.environ['V2_AUTO_PROCESS_PROPOSALS_PHASE0'] = 'true'
os.environ['V2_LLM_ENABLED'] = 'true'
os.environ['V2_LLM_USE_LEGACY_INTERFACE'] = 'false'
os.environ['V2_LLM_REPAIR_ON_VALIDATION_ERROR'] = 'true'
os.environ['V2_LLM_PROVIDER'] = 'openai'
os.environ['V2_LLM_ENDPOINT'] = 'https://api.openai.com/v1/chat/completions'
os.environ['V2_LLM_API_KEY'] = V2_LLM_API_KEY
os.environ['V2_LLM_MODEL'] = V2_LLM_MODEL
if GEMINI_API_KEY.strip():
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    os.environ['V2_LLM_FALLBACK_PROVIDER'] = 'gemini'
    os.environ['V2_LLM_FALLBACK_MODEL'] = V2_LLM_FALLBACK_MODEL
os.environ['V2_LLM_CONFIG_FILE'] = 'config/llm_settings.json'
os.environ['V2_LLM_PROMPT_TEMPLATE_FILE'] = 'prompts/generate_new_models.txt'
os.environ['V2_LLM_FIX_ERROR_PROMPT_FILE'] = 'prompts/fix_model_error.txt'
os.environ['V2_LLM_ARCHITECTURE_GUIDE_FILE'] = 'prompts/instruccions.md'
os.environ['V2_PROMPT_REFERENCE_MODEL_PATH'] = 'models/base/model_exemple_complex_v1.json'
print('Variables base configurades.')

In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

root = REPO_ROOT
repo = root
zip_path = Path(DATA_ZIP_PATH)
if not zip_path.exists():
    raise FileNotFoundError(f'ZIP no trobat: {zip_path}')

extract_root = repo / 'data' / 'runtime_drive'
target_dir = extract_root / DATASET_RUNTIME_NAME
marker = target_dir / '.extract_complete'
if CLEAN_EXTRACT and target_dir.exists():
    shutil.rmtree(target_dir)
if not marker.exists():
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(target_dir)
    marker.write_text('ok', encoding='utf-8')

canonical_required = {
    'entrada_valors.csv', 'entrada_extra.csv', 'min.csv', 'max.csv',
    'sortida_min.csv', 'sortida_max.csv', 'sortida_tb.csv', 'sortida_sl.csv',
    'sortida_sn.csv', 'sortida_valors.csv'
}
compat_pairs = [
    ('sortida_min_7d.csv', 'sortida_min.csv'),
    ('sortida_max_7d.csv', 'sortida_max.csv'),
    ('sortida_valors_7d.csv', 'sortida_valors.csv'),
]

candidate_dirs = [target_dir] + [p for p in target_dir.rglob('*') if p.is_dir()]
dataset_dir = None
for candidate in candidate_dirs:
    for src_name, dst_name in compat_pairs:
        src = candidate / src_name
        dst = candidate / dst_name
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
    existing = {p.name for p in candidate.glob('*.csv')}
    if canonical_required.issubset(existing):
        dataset_dir = candidate
        break

if dataset_dir is None:
    raise FileNotFoundError('No s\'ha trobat cap carpeta amb tots els CSV requerits dins del ZIP.')

base_cfg_path = repo / 'configs' / 'experiment_config.json'
runtime_cfg_path = repo / 'configs' / 'experiment_config.drive_runtime.json'
cfg = json.loads(base_cfg_path.read_text(encoding='utf-8'))
cfg['data_dir'] = dataset_dir.relative_to(repo).as_posix()
runtime_cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')

os.environ['V2_LLM_EXPERIMENT_CONFIG_FILE'] = 'configs/experiment_config.drive_runtime.json'
os.environ['V2_LEGACY_EXPERIMENT_CONFIG_PATH'] = 'configs/experiment_config.drive_runtime.json'

summary = {
    'zip_path': str(zip_path),
    'extract_root': str(target_dir),
    'dataset_dir': str(dataset_dir),
    'runtime_config': str(runtime_cfg_path),
    'data_dir_in_config': cfg['data_dir'],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
from pathlib import Path
import json

root = REPO_ROOT
required_paths = [
    'colab-worker/run_worker_loop.py',
    'colab-worker/run_trainer.py',
    'colab-worker/src/trainer.py',
    'colab-worker/src/engine.py',
    'ops/scripts/probe_api_prefix.py',
    'ops/scripts/go_no_go_check.py',
    'configs/experiment_config.drive_runtime.json',
    'models/base/model_exemple_complex_v1.json'
]
missing = [p for p in required_paths if not (root / p).exists()]
print(json.dumps({'ok': len(missing) == 0, 'missing': missing}, ensure_ascii=False, indent=2))
if missing:
    raise FileNotFoundError('Falten fitxers requerits per al runtime.')

!python ops/scripts/probe_api_prefix.py
!python ops/scripts/go_no_go_check.py

## Execucio
Quan el monitor del servidor tingui una `execution_request` pendent, executa la cel·la seguent i deixa el notebook obert.

In [ ]:
!python colab-worker/run_worker_loop.py